In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .master("local[*]")
    .appName("Spark Streaming")
    .getOrCreate()
)



In [2]:
spark

In [8]:
df_raw = spark.read.format("text").load("data/example.txt")

df_raw.printSchema()


root
 |-- value: string (nullable = true)



In [9]:
df_raw.show()

+--------------------+
|               value|
+--------------------+
|this is a very fi...|
+--------------------+



In [10]:
from pyspark.sql.functions import split

df_words = df_raw.withColumn("words", split("value", " "))

In [11]:
df_words.show()

+--------------------+--------------------+
|               value|               words|
+--------------------+--------------------+
|this is a very fi...|[this, is, a, ver...|
+--------------------+--------------------+



In [19]:
from pyspark.sql.functions import explode

df_explode = df_words.withColumn("word", explode("words")).drop("value", "words")

In [20]:
df_explode.show()

+-------+
|   word|
+-------+
|   this|
|     is|
|      a|
|   very|
|  first|
|   test|
|    iam|
|  doing|
|    for|
|    the|
|  kafka|
|  spark|
|jupyter|
|   with|
| docker|
|    for|
|  kafka|
|  spark|
|jupyter|
+-------+



In [21]:
from pyspark.sql.functions import count, lit
df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("cnt"))

In [22]:
df_agg.show()

+-------+---+
|   word|cnt|
+-------+---+
|    for|  2|
|  kafka|  2|
|   with|  1|
| docker|  1|
|     is|  1|
|jupyter|  2|
|    iam|  1|
|  doing|  1|
|  spark|  2|
|    the|  1|
|   very|  1|
|      a|  1|
|   this|  1|
|  first|  1|
|   test|  1|
+-------+---+

